In [24]:
import pandas as pd
import numpy as np
from chembl_webresource_client.new_client import new_client
import re

**DATA DONE**

ligand_master:

binding summaries & SMILES feautures

model_df_cleaned:

merged ids with ligands column

model_df_ml_ready!!!

In [38]:
cptac_model_df = pd.read_csv("data/cptac_model_df.csv")
mutation_df = pd.read_csv("data/mutation_features_cleaned.csv")
binding_df = pd.read_csv("data/egfr_ligand_binding.csv")
smiles_df = pd.read_csv("data/ligand_smiles_features.csv")

print("cptac_model_df:", cptac_model_df.shape)
print("mutation_df:", mutation_df.shape)
print("binding_df:", binding_df.shape)
print("smiles_df:", smiles_df.shape)

cptac_model_df: (207, 12)
mutation_df: (70, 10)
binding_df: (29, 6)
smiles_df: (3, 11)


In [26]:
#mutation labels
def label_egfr_hotspot(mutation_value: str) -> str:
    m = str(mutation_value).upper().strip()

    if (
        "EXON 19" in m
        or "19DEL" in m
        or "DEL19" in m
        or re.search(r"E\d+_A\d+DEL", m)
        or re.search(r"L\d+_A\d+DEL", m)
        or re.search(r"L\d+_T\d+DEL", m)
        or "DELINS" in m
    ):
        return "exon19del"
    elif "L858R" in m:
        return "L858R"
    elif "T790M" in m:
        return "T790M"
    elif "C797S" in m:
        return "C797S"
    elif "G719" in m:
        return "G719X"
    elif "L861Q" in m:
        return "L861Q"
    elif "S768I" in m:
        return "S768I"
    elif "EXON 20" in m or "INS" in m or "DUP" in m:
        return "exon20_alteration"
    else:
        return "other"

mutation_df["mutation"] = mutation_df["mutation"].astype(str).str.strip()
mutation_df["egfr_hotspot_label"] = mutation_df["mutation"].apply(label_egfr_hotspot)
mutation_df["is_egfr_hotspot"] = mutation_df["egfr_hotspot_label"].ne("other").astype(int)

patient_mut = (
    mutation_df[["patient_id", "mutation", "egfr_hotspot_label", "is_egfr_hotspot"]]
    .drop_duplicates(subset=["patient_id"])
)

print(patient_mut.head())
print(patient_mut["egfr_hotspot_label"].value_counts(dropna=False))

        patient_id                mutation egfr_hotspot_label  is_egfr_hotspot
0  TCGA-05-4382-01             R222L E545Q              other                0
1  TCGA-05-4402-01  T751_I759delinsN I759N          exon19del                1
2  TCGA-05-4410-01                   R377S              other                0
3  TCGA-05-5423-01             L833F L861Q              L861Q                1
4  TCGA-17-Z026-01                   G721V              other                0
egfr_hotspot_label
exon19del            24
L858R                23
other                14
L861Q                 3
exon20_alteration     3
G719X                 3
Name: count, dtype: int64


In [27]:
#ligand binding raws from chembl ligand source
binding_df["standard_value"] = pd.to_numeric(binding_df["standard_value"], errors="coerce")
binding_df["p_binding"] = pd.to_numeric(binding_df["p_binding"], errors="coerce")

binding_df = binding_df.dropna(subset=["ligand", "protein", "binding_type", "standard_value", "p_binding"]).copy()
binding_df = binding_df[binding_df["standard_value"] > 0].copy()

ligand_binding_summary = (
    binding_df
    .groupby(["ligand", "protein", "binding_type"], as_index=False)
    .agg(
        median_binding_nM=("standard_value", "median"),
        median_p_binding=("p_binding", "median"),
        n_measurements=("standard_value", "count")
    )
)

ligand_binding_summary["standard_units"] = "nM"

print(ligand_binding_summary.shape)
print(ligand_binding_summary.head())

#SMLIES
smiles_df = smiles_df.drop_duplicates(subset=["ligand"]).copy()

print(smiles_df.columns.tolist())
print(smiles_df[["ligand", "pref_name", "canonical_smiles"]].head())

(3, 7)
          ligand    protein binding_type  median_binding_nM  median_p_binding  \
0  CHEMBL1079742  CHEMBL203         IC50             42.500          7.489405   
1      CHEMBL941  CHEMBL203         IC50          50000.055          6.979304   
2      CHEMBL941  CHEMBL203           Kd          10000.000          5.000000   

   n_measurements standard_units  
0               4             nM  
1               2             nM  
2              23             nM  
['ligand', 'pref_name', 'canonical_smiles', 'MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds', 'RingCount', 'HeavyAtomCount']
          ligand                pref_name  \
0      CHEMBL941                 IMATINIB   
1  CHEMBL1079742  ERLOTINIB HYDROCHLORIDE   
2  CHEMBL2105758     AVATROMBOPAG MALEATE   

                                    canonical_smiles  
0  Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc...  
1      C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1.Cl  
2  O=C(Nc1nc(-c2cc(Cl)cs2)c

In [28]:
#master ligand + drug binding 
ligand_master = ligand_binding_summary.merge(
    smiles_df,
    on="ligand",
    how="left",
    validate="many_to_one"
)

print("ligand_master shape:", ligand_master.shape)
print(ligand_master.head())

ligand_master.to_csv("data/ligand_master.csv", index=False)

ligand_master shape: (3, 17)
          ligand    protein binding_type  median_binding_nM  median_p_binding  \
0  CHEMBL1079742  CHEMBL203         IC50             42.500          7.489405   
1      CHEMBL941  CHEMBL203         IC50          50000.055          6.979304   
2      CHEMBL941  CHEMBL203           Kd          10000.000          5.000000   

   n_measurements standard_units                pref_name  \
0               4             nM  ERLOTINIB HYDROCHLORIDE   
1               2             nM                 IMATINIB   
2              23             nM                 IMATINIB   

                                    canonical_smiles    MolWt  MolLogP   TPSA  \
0      C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1.Cl  429.904  3.82690  74.73   
1  Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc...  493.615  4.59032  86.28   
2  Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc...  493.615  4.59032  86.28   

   NumHDonors  NumHAcceptors  NumRotatableBonds  RingCount  HeavyAtomCount  

In [29]:
#merge into cptac_model

cptac_model_df = cptac_model_df.drop_duplicates(subset=["patient_id"]).copy()

cptac_model_df = cptac_model_df.merge(
    patient_mut,
    on="patient_id",
    how="left",
    validate="one_to_one"
)

cptac_model_df["mutation"] = cptac_model_df["mutation"].fillna("unknown")
cptac_model_df["egfr_hotspot_label"] = cptac_model_df["egfr_hotspot_label"].fillna("other")
cptac_model_df["is_egfr_hotspot"] = cptac_model_df["is_egfr_hotspot"].fillna(0)

print(cptac_model_df.shape)
print(cptac_model_df.head())

(106, 15)
  patient_id  EGFR_PROTEIN  EGFR_RNA  EGFR_activity_mean cohort batch_domain  \
0  C3L-00001     28.192875     17.07                   1  CPTAC        CPTAC   
1  C3L-00009     25.219585     11.86                   1  CPTAC        CPTAC   
2  C3L-00080     25.238803     12.58                   1  CPTAC        CPTAC   
3  C3L-00083     25.041583     10.94                   1  CPTAC        CPTAC   
4  C3L-00093     24.469511     12.78                   1  CPTAC        CPTAC   

   phospho_Y1016  phospho_Y1069  phospho_Y1092  phospho_Y1110  phospho_Y1172  \
0            NaN            NaN            NaN            NaN      20.075879   
1            NaN            NaN      16.931486            NaN      18.767056   
2            NaN            NaN            NaN            NaN      18.026248   
3            NaN            NaN            NaN            NaN            NaN   
4            NaN            NaN            NaN            NaN            NaN   

   EGFR_expression_score mut

In [30]:
#matching patients
cptac_model_df["key"] = 1
ligand_master["key"] = 1

model_df = cptac_model_df.merge(
    ligand_master,
    on="key",
    how="inner"
).drop(columns=["key"])

print("model_df shape:", model_df.shape)
print(model_df.head())


model_df shape: (318, 32)
  patient_id  EGFR_PROTEIN  EGFR_RNA  EGFR_activity_mean cohort batch_domain  \
0  C3L-00001     28.192875     17.07                   1  CPTAC        CPTAC   
1  C3L-00001     28.192875     17.07                   1  CPTAC        CPTAC   
2  C3L-00001     28.192875     17.07                   1  CPTAC        CPTAC   
3  C3L-00009     25.219585     11.86                   1  CPTAC        CPTAC   
4  C3L-00009     25.219585     11.86                   1  CPTAC        CPTAC   

   phospho_Y1016  phospho_Y1069  phospho_Y1092  phospho_Y1110  ...  \
0            NaN            NaN            NaN            NaN  ...   
1            NaN            NaN            NaN            NaN  ...   
2            NaN            NaN            NaN            NaN  ...   
3            NaN            NaN      16.931486            NaN  ...   
4            NaN            NaN      16.931486            NaN  ...   

                 pref_name                                   canonical_s

In [32]:
#data cleaning
model_df = model_df.dropna(subset=["patient_id", "ligand"]).copy()

numeric_cols = model_df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in model_df.select_dtypes(exclude=["number"]).columns if c not in ["patient_id"]]

for col in numeric_cols:
    model_df[col] = model_df[col].fillna(model_df[col].median())

for col in categorical_cols:
    model_df[col] = model_df[col].fillna("unknown")

#sanity check since it was A LOT OF NANS IN DATA!!
#print(model_df.isna().sum().sort_values(ascending=False).head(20)) 


#consistent mutation and druf naming
model_df["pref_name"] = model_df["pref_name"].astype(str).str.strip().str.lower()
model_df["binding_type"] = model_df["binding_type"].astype(str).str.strip().str.upper()
model_df["egfr_hotspot_label"] = model_df["egfr_hotspot_label"].astype(str).str.strip()

print(model_df[["ligand", "pref_name", "binding_type", "egfr_hotspot_label"]].head())

          ligand                pref_name binding_type egfr_hotspot_label
0  CHEMBL1079742  erlotinib hydrochloride         IC50              other
1      CHEMBL941                 imatinib         IC50              other
2      CHEMBL941                 imatinib           KD              other
3  CHEMBL1079742  erlotinib hydrochloride         IC50              other
4      CHEMBL941                 imatinib         IC50              other


In [35]:
#mutation-binding columns ffrom model 
model_df["mutation_binding_context"] = (
    model_df["egfr_hotspot_label"].astype(str) + "_" + model_df["binding_type"].astype(str)
)

model_df["protein_ligand_pair"] = (
    model_df["protein"].astype(str) + "_" + model_df["ligand"].astype(str)
)

print(model_df[["mutation_binding_context", "protein_ligand_pair"]].head())
model_df.to_csv("data/model_df_clean.csv", index=False)

  mutation_binding_context      protein_ligand_pair
0               other_IC50  CHEMBL203_CHEMBL1079742
1               other_IC50      CHEMBL203_CHEMBL941
2                 other_KD      CHEMBL203_CHEMBL941
3               other_IC50  CHEMBL203_CHEMBL1079742
4               other_IC50      CHEMBL203_CHEMBL941


In [ ]:
#hot spot distributions
categorical_to_encode = [
    "cohort",
    "batch_domain",
    "mutation",
    "egfr_hotspot_label",
    "binding_type",
    "pref_name",
    "mutation_binding_context",
    "protein_ligand_pair"
]

existing_catg = [c for c in categorical_to_encode if c in model_df.columns]

ml_ready_df = pd.get_dummies(
    model_df,
    columns=existing_catg,
    drop_first=False
)

print("ml_ready_df shape:", ml_ready_df.shape)
print(ml_ready_df.head())



#checkpoint done?
ml_ready_df.to_csv("data/model_df_ml_ready.csv", index=False)

ml_ready_df shape: (318, 38)
  patient_id  EGFR_PROTEIN  EGFR_RNA  EGFR_activity_mean  phospho_Y1016  \
0  C3L-00001     28.192875     17.07                   1      13.072192   
1  C3L-00001     28.192875     17.07                   1      13.072192   
2  C3L-00001     28.192875     17.07                   1      13.072192   
3  C3L-00009     25.219585     11.86                   1      13.072192   
4  C3L-00009     25.219585     11.86                   1      13.072192   

   phospho_Y1069  phospho_Y1092  phospho_Y1110  phospho_Y1172  \
0      12.604645      17.767915      10.583122      20.075879   
1      12.604645      17.767915      10.583122      20.075879   
2      12.604645      17.767915      10.583122      20.075879   
3      12.604645      16.931486      10.583122      18.767056   
4      12.604645      16.931486      10.583122      18.767056   

   EGFR_expression_score  ...  mutation_unknown egfr_hotspot_label_other  \
0              20.075879  ...              True      